<a href="https://colab.research.google.com/github/MariaMuu/Creatine-Simulation-Analysis/blob/main/Creatine_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Import important libraries

In [46]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 800  # base sample size

##Basic demographic features

In [47]:
age = np.random.randint(18, 65, n)

sex = np.random.choice(["male", "female"], size=n)

# encode sex for model-friendly noise later
sex_num = np.where(sex == "male", 1, 0)

# Add random Greek names
import random

male_first_names = ["Georgios", "Dimitrios", "Konstantinos", "Ioannis", "Nikolaos", "Christos", "Panagiotis", "Vasileios", "Spyridon", "Anastasios"]
female_first_names = ["Maria", "Eleni", "Aikaterini", "Vasiliki", "Sofia", "Anastasia", "Evangelia", "Alexandra", "Dimitra", "Anna"]
last_names = ["Papadopoulos", "Pappas", "Karagiannis", "Vasileiou", "Georgiou", "Nikolaou", "Konstantinidis", "Dimopoulos", "Adamidis", "Ioannidis"]

first_name = np.array([random.choice(male_first_names) if s == "male" else random.choice(female_first_names) for s in sex])
last_name = np.array([random.choice(last_names) for _ in range(n)])
name = np.array([f"{f} {l}" for f, l in zip(first_name, last_name)])

In [48]:

# Creatine dose (5g - 30g)

dose = np.random.uniform(5, 30, n)

##Synthetic effects - Simulation rules

In [49]:
# Age effects (older = slightly worse baseline)
age_effect = (age - 18) * -0.03

# Sex effect (tiny arbitrary difference for realism)
sex_effect = sex_num * 0.5

# Dose nonlinear effect (diminishing returns)
dose_effect_sleep = 0.10 * dose - 0.002 * (dose ** 2)
dose_effect_mood = 1.2 * dose - 0.015 * (dose ** 2)
dose_effect_cog = 0.6 * dose - 0.008 * (dose ** 2)

Outcomes

In [50]:
sleep_score = (
    6
    + dose_effect_sleep
    + age_effect * 0.5
    + sex_effect * 0.2
    + np.random.normal(0, 1.0, n)
)

mood_score = (
    50
    + dose_effect_mood
    + age_effect * 2
    + sex_effect * 1
    + np.random.normal(0, 8, n)
)

cognition_score = (
    100
    + dose_effect_cog
    + age_effect * 1.5
    + sex_effect * 1.5
    + np.random.normal(0, 5, n)
)

In [51]:
# ----------------------------
# 5. Build dataframe
# ----------------------------

df = pd.DataFrame({
    "name": name,
    "age": age,
    "sex": sex,
    "dose_g": dose,
    "sleep_score": sleep_score,
    "mood_score": mood_score,
    "cognition_score": cognition_score
})

In [52]:
# ----------------------------
# 6. Inject REAL-WORLD messiness
# ----------------------------

# Missing values (5%)
for col in df.columns:
    df.loc[df.sample(frac=0.05).index, col] = np.nan

# Duplicate rows
df = pd.concat([df, df.sample(30)], ignore_index=True)

# Outliers (bad measurement / extreme users)
outliers = pd.DataFrame({
    "age": [5, 120, 999],
    "sex": ["male", "female", "male"],
    "dose_g": [100, -10, 250],
    "sleep_score": [25, -5, 60],
    "mood_score": [300, -150, 800],
    "cognition_score": [400, -100, 900]
})

df = pd.concat([df, outliers], ignore_index=True)

# Corrupt some values (sensor / entry errors)
idx = df.sample(15).index
df.loc[idx, "mood_score"] = df.loc[idx, "mood_score"] * 2

# Introduce varied casing/single-letter for sex
alternative_sex_values = ["MALE", "FEMALE", "f", "m", "F", "M"]
num_to_change = min(10, len(df)) # Change up to 10 values, or fewer if df is small

if num_to_change > 0:
    indices_to_change = np.random.choice(df.index, num_to_change, replace=False)
    for idx in indices_to_change:
        df.loc[idx, "sex"] = random.choice(alternative_sex_values)

# Shuffle rows
df = df.sample(frac=1).reset_index(drop=True)

# Change data type
df['age'] = df['age'].astype(str)

In [53]:
# ----------------------------
# 7. Save dataset
# ----------------------------

df.to_csv("synthetic_creatine_study.csv", index=False)

print(df.head())
print("\nDataset shape:", df.shape)

                     name   age     sex     dose_g  sleep_score  mood_score  \
0  Konstantinos Vasileiou  28.0    male  14.285214     7.505528   67.122028   
1      Nikolaos Vasileiou  39.0    male  13.078266     5.341051   59.524379   
2        Ioannis Georgiou  36.0    male   5.446547     6.274310   59.882594   
3             Anna Pappas  30.0     NaN  25.468375     5.943014   65.241407   
4  Dimitra Konstantinidis  50.0  female  10.258739     6.839209   66.517821   

   cognition_score  
0       115.977027  
1       106.545224  
2        92.989228  
3       113.689930  
4        96.246922  

Dataset shape: (833, 7)
